# LangGraph and LangSmith - Agentic RAG Powered by LangChain

In the following notebook we'll complete the following tasks:

- 🤝 Breakout Room #1:

  1. Install required libraries
  2. Set Environment Variables
  3. Creating our Tool Belt
  4. Creating Our State
  5. Creating and Compiling A Graph!

- 🤝 Breakout Room #2:
  1. Evaluating the LangGraph Application with LangSmith
  2. Adding Helpfulness Check and "Loop" Limits
  3. LangGraph for the "Patterns" of GenAI


# 🤝 Breakout Room #1


## Part 1: LangGraph - Building Cyclic Applications with LangChain

LangGraph is a tool that leverages LangChain Expression Language to build coordinated multi-actor and stateful applications that includes cyclic behaviour.

### Why Cycles?

In essence, we can think of a cycle in our graph as a more robust and customizable loop. It allows us to keep our application agent-forward while still giving the powerful functionality of traditional loops.

Due to the inclusion of cycles over loops, we can also compose rather complex flows through our graph in a much more readable and natural fashion. Effectively allowing us to recreate application flowcharts in code in an almost 1-to-1 fashion.

### Why LangGraph?

Beyond the agent-forward approach - we can easily compose and combine traditional "DAG" (directed acyclic graph) chains with powerful cyclic behaviour due to the tight integration with LCEL. This means it's a natural extension to LangChain's core offerings!


## Task 1: Dependencies


## Task 2: Environment Variables

We'll want to set our OpenAI, Tavily, and LangSmith API keys along with our LangSmith environment variables.


In [1]:
import os
import getpass

os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API Key:")

In [2]:
os.environ["TAVILY_API_KEY"] = getpass.getpass("TAVILY_API_KEY")

In [3]:
from uuid import uuid4

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = f"AIE8 - LangGraph - {uuid4().hex[0:8]}"
os.environ["LANGCHAIN_API_KEY"] = getpass.getpass("LangSmith API Key: ")

## Task 3: Creating our Tool Belt

As is usually the case, we'll want to equip our agent with a toolbelt to help answer questions and add external knowledge.

There's a tonne of tools in the [LangChain Community Repo](https://github.com/langchain-ai/langchain-community/tree/main/libs/community) but we'll stick to a couple just so we can observe the cyclic nature of LangGraph in action!

We'll leverage:

- [Tavily Search Results](https://github.com/langchain-ai/langchain-community/blob/main/libs/community/langchain_community/tools/tavily_search/tool.py)
- [Arxiv](https://github.com/langchain-ai/langchain-community/blob/main/libs/community/langchain_community/tools/arxiv/tool.py)


#### 🏗️ Activity #1:

Please add the tools to use into our toolbelt.

> NOTE: Each tool in our toolbelt should be a method.


In [4]:
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_community.tools.arxiv.tool import ArxivQueryRun

tavily_tool = TavilySearchResults(max_results=5)

tool_belt = [
    tavily_tool,
    ArxivQueryRun(),
]

/var/folders/m2/13x6wtqs1c52469w0mptfrdc0000gn/T/ipykernel_81939/1203815797.py:4: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-tavily package and should be used instead. To use it run `pip install -U :class:`~langchain-tavily` and import as `from :class:`~langchain_tavily import TavilySearch``.
  tavily_tool = TavilySearchResults(max_results=5)


### Model

Now we can set-up our model! We'll leverage the familiar OpenAI model suite for this example - but it's not _necessary_ to use with LangGraph. LangGraph supports all models - though you might not find success with smaller models - as such, they recommend you stick with:

- OpenAI's GPT-3.5 and GPT-4
- Anthropic's Claude
- Google's Gemini

> NOTE: Because we're leveraging the OpenAI function calling API - we'll need to use OpenAI _for this specific example_ (or any other service that exposes an OpenAI-style function calling API.


In [7]:
from langchain_openai import ChatOpenAI

model = ChatOpenAI(model="gpt-4.1-nano", temperature=0)

Now that we have our model set-up, let's "put on the tool belt", which is to say: We'll bind our LangChain formatted tools to the model in an OpenAI function calling format.


In [8]:
model = model.bind_tools(tool_belt)

#### ❓ Question #1:

How does the model determine which tool to use?

**Answer:** The model determines which tool to use through a combination of several mechanisms:

1. **Function Calling API**: When we bind tools to the model using `model.bind_tools(tool_belt)`, the model receives tool schemas that describe each available tool's name, parameters, and purpose. The model uses this information to decide which tool is most appropriate for the current task.

2. **Context Analysis**: The model analyzes the user's query and the conversation context to understand what information is needed. It then matches this need against the available tools' capabilities.

3. **Tool Selection Logic**: The model doesn't just randomly pick a tool - it uses its training on function calling to:

   - Parse the user's intent
   - Identify which tool can best fulfill that intent
   - Generate the appropriate function call with correct parameters

4. **Conditional Edge Logic**: The `should_continue` function checks if the model's response contains `tool_calls` in the `additional_kwargs`. If it does, the state flows to the "action" node to execute the tool. If not, it ends the conversation.

5. **Iterative Refinement**: The model can use multiple tools in sequence, with each tool's output informing the next decision about whether additional tools are needed.

The prompt itself doesn't directly determine tool selection - rather, it's the model's understanding of the task combined with the tool schemas that drives the decision-making process.


## Task 4: Putting the State in Stateful

Earlier we used this phrasing:

`coordinated multi-actor and stateful applications`

So what does that "stateful" mean?

To put it simply - we want to have some kind of object which we can pass around our application that holds information about what the current situation (state) is. Since our system will be constructed of many parts moving in a coordinated fashion - we want to be able to ensure we have some commonly understood idea of that state.

LangGraph leverages a `StatefulGraph` which uses an `AgentState` object to pass information between the various nodes of the graph.

There are more options than what we'll see below - but this `AgentState` object is one that is stored in a `TypedDict` with the key `messages` and the value is a `Sequence` of `BaseMessages` that will be appended to whenever the state changes.

Let's think about a simple example to help understand exactly what this means (we'll simplify a great deal to try and clearly communicate what state is doing):

1. We initialize our state object:

- `{"messages" : []}`

2. Our user submits a query to our application.

- New State: `HumanMessage(#1)`
- `{"messages" : [HumanMessage(#1)}`

3. We pass our state object to an Agent node which is able to read the current state. It will use the last `HumanMessage` as input. It gets some kind of output which it will add to the state.

- New State: `AgentMessage(#1, additional_kwargs {"function_call" : "WebSearchTool"})`
- `{"messages" : [HumanMessage(#1), AgentMessage(#1, ...)]}`

4. We pass our state object to a "conditional node" (more on this later) which reads the last state to determine if we need to use a tool - which it can determine properly because of our provided object!


In [9]:
from typing import TypedDict, Annotated
from langgraph.graph.message import add_messages
import operator
from langchain_core.messages import BaseMessage

class AgentState(TypedDict):
  messages: Annotated[list, add_messages]

## Task 5: It's Graphing Time!

Now that we have state, and we have tools, and we have an LLM - we can finally start making our graph!

Let's take a second to refresh ourselves about what a graph is in this context.

Graphs, also called networks in some circles, are a collection of connected objects.

The objects in question are typically called nodes, or vertices, and the connections are called edges.

Let's look at a simple graph.

![image](https://i.imgur.com/2NFLnIc.png)

Here, we're using the coloured circles to represent the nodes and the yellow lines to represent the edges. In this case, we're looking at a fully connected graph - where each node is connected by an edge to each other node.

If we were to think about nodes in the context of LangGraph - we would think of a function, or an LCEL runnable.

If we were to think about edges in the context of LangGraph - we might think of them as "paths to take" or "where to pass our state object next".

Let's create some nodes and expand on our diagram.

> NOTE: Due to the tight integration with LCEL - we can comfortably create our nodes in an async fashion!


In [10]:
from langgraph.prebuilt import ToolNode

def call_model(state):
  messages = state["messages"]
  response = model.invoke(messages)
  return {"messages" : [response]}

tool_node = ToolNode(tool_belt)

Now we have two total nodes. We have:

- `call_model` is a node that will...well...call the model
- `tool_node` is a node which can call a tool

Let's start adding nodes! We'll update our diagram along the way to keep track of what this looks like!


In [11]:
from langgraph.graph import StateGraph, END

uncompiled_graph = StateGraph(AgentState)

uncompiled_graph.add_node("agent", call_model)
uncompiled_graph.add_node("action", tool_node)

Let's look at what we have so far:

![image](https://i.imgur.com/md7inqG.png)


Next, we'll add our entrypoint. All our entrypoint does is indicate which node is called first.


In [12]:
uncompiled_graph.set_entry_point("agent")

![image](https://i.imgur.com/wNixpJe.png)


Now we want to build a "conditional edge" which will use the output state of a node to determine which path to follow.

We can help conceptualize this by thinking of our conditional edge as a conditional in a flowchart!

Notice how our function simply checks if there is a "function_call" kwarg present.

Then we create an edge where the origin node is our agent node and our destination node is _either_ the action node or the END (finish the graph).

It's important to highlight that the dictionary passed in as the third parameter (the mapping) should be created with the possible outputs of our conditional function in mind. In this case `should_continue` outputs either `"end"` or `"continue"` which are subsequently mapped to the action node or the END node.


In [13]:
def should_continue(state):
  last_message = state["messages"][-1]

  if last_message.tool_calls:
    return "action"

  return END

uncompiled_graph.add_conditional_edges(
    "agent",
    should_continue
)

Let's visualize what this looks like.

![image](https://i.imgur.com/8ZNwKI5.png)


Finally, we can add our last edge which will connect our action node to our agent node. This is because we _always_ want our action node (which is used to call our tools) to return its output to our agent!


In [14]:
uncompiled_graph.add_edge("action", "agent")

Let's look at the final visualization.

![image](https://i.imgur.com/NWO7usO.png)


All that's left to do now is to compile our workflow - and we're off!


In [15]:
simple_agent_graph = uncompiled_graph.compile()

#### ❓ Question #2:

Is there any specific limit to how many times we can cycle?

**Answer:** No, there is no built-in limit to how many times a LangGraph can cycle by default. The graph will continue cycling until one of the conditional edges returns `END` or the graph reaches a terminal state.

If not, how could we impose a limit to the number of cycles?

**Answer:** There are several ways to impose limits on the number of cycles:

1. **Message Count Limit**: Check the length of the messages array in the state:

   ```python
   if len(state["messages"]) > 10:  # Limit to 10 messages
       return "END"
   ```

2. **Iteration Counter**: Add a counter to the state and increment it each cycle:

   ```python
   class AgentState(TypedDict):
       messages: Annotated[list, add_messages]
       iteration_count: int

   # In conditional function:
   if state["iteration_count"] > 5:  # Limit to 5 iterations
       return "END"
   ```

3. **Time-based Limits**: Use timestamps to limit execution time:

   ```python
   import time

   class AgentState(TypedDict):
       messages: Annotated[list, add_messages]
       start_time: float

   # In conditional function:
   if time.time() - state["start_time"] > 300:  # 5 minute limit
       return "END"
   ```

4. **Content-based Termination**: Use AI to determine when to stop (as shown later in this notebook):

   ```python
   # Check if response is helpful enough to end
   if helpfulness_score > threshold:
       return "end"
   ```

5. **Tool Usage Limits**: Limit the number of tool calls:
   ```python
   tool_call_count = sum(1 for msg in state["messages"] if hasattr(msg, 'tool_calls') and msg.tool_calls)
   if tool_call_count > 3:  # Limit to 3 tool calls
       return "END"
   ```

The key is to modify the conditional edge function (`should_continue` or similar) to include your chosen termination criteria.


## Using Our Graph

Now that we've created and compiled our graph - we can call it _just as we'd call any other_ `Runnable`!

Let's try out a few examples to see how it fairs:


In [16]:
from langchain_core.messages import HumanMessage

inputs = {"messages" : [HumanMessage(content="How are technical professionals using AI to improve their work?")]}

async for chunk in simple_agent_graph.astream(inputs, stream_mode="updates"):
    for node, values in chunk.items():
        print(f"Receiving update from node: '{node}'")
        print(values["messages"])
        print("\n\n")

Receiving update from node: 'agent'
[AIMessage(content='Technical professionals are using AI in various ways to enhance their work, including automating repetitive tasks, improving data analysis, developing smarter algorithms, enhancing cybersecurity, and creating innovative products. Would you like specific examples from particular fields such as software development, data science, engineering, or cybersecurity?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 57, 'prompt_tokens': 163, 'total_tokens': 220, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4.1-nano-2025-04-14', 'system_fingerprint': 'fp_7c233bf9d1', 'id': 'chatcmpl-CJ7cPqZYlmdCQRg2ugaowhWjubT3w', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--1ccc8b9c-1641-4a05-b99e-b4f

Let's look at what happened:

1. Our state object was populated with our request
2. The state object was passed into our entry point (agent node) and the agent node added an `AIMessage` to the state object and passed it along the conditional edge
3. The conditional edge received the state object, found the "tool_calls" `additional_kwarg`, and sent the state object to the action node
4. The action node added the response from the OpenAI function calling endpoint to the state object and passed it along the edge to the agent node
5. The agent node added a response to the state object and passed it along the conditional edge
6. The conditional edge received the state object, could not find the "tool_calls" `additional_kwarg` and passed the state object to END where we see it output in the cell above!

Now let's look at an example that shows a multiple tool usage - all with the same flow!


In [17]:
inputs = {"messages" : [HumanMessage(content="Search Arxiv for the A Comprehensive Survey of Deep Research paper, then search each of the authors to find out where they work now using Tavily!")]}

async for chunk in simple_agent_graph.astream(inputs, stream_mode="updates"):
    for node, values in chunk.items():
        print(f"Receiving update from node: '{node}'")
        if node == "action":
          print(f"Tool Used: {values['messages'][0].name}")
        print(values["messages"])

        print("\n\n")

Receiving update from node: 'agent'
[AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_lnUkMNthmPLUOerH97iZj3LG', 'function': {'arguments': '{"query": "A Comprehensive Survey of Deep Research"}', 'name': 'arxiv'}, 'type': 'function'}, {'id': 'call_ABnRHmKkLha5re1AevbEzH04', 'function': {'arguments': '{"query": "A Comprehensive Survey of Deep Research paper"}', 'name': 'tavily_search_results_json'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 59, 'prompt_tokens': 182, 'total_tokens': 241, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4.1-nano-2025-04-14', 'system_fingerprint': 'fp_7c233bf9d1', 'id': 'chatcmpl-CJ7cWtwvZDe0zP55Q5CTkAOckk1KR', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run--a5442368-4205-46

#### 🏗️ Activity #2:

Please write out the steps the agent took to arrive at the correct answer.

**Answer:**

1. **Initial Query Processing**: The agent received the user's query: "Search Arxiv for the A Comprehensive Survey of Deep Research paper, then search each of the authors to find out where they work now using Tavily!"

2. **Tool Selection and Planning**: The agent analyzed the query and determined it needed to use two tools:
   - Arxiv tool to search for the specific research paper
   - Tavily search tool to find information about the authors' current workplaces

3. **Simultaneous Tool Calls**: The agent made two tool calls in a single response:
   - **Arxiv search**: Searched for "A Comprehensive Survey of Deep Research" to find the paper
   - **Tavily search**: Searched for "A Comprehensive Survey of Deep Research paper" to gather additional context

4. **Tool Results Processing**: The action node executed both tools and received results:
   - Arxiv returned information about the paper including authors (Renjun Xu, Jingwen Peng), publication date (2025-06-14), and summary
   - Tavily provided additional search results about the paper

5. **Response Generation**: The agent processed the tool results and generated a comprehensive response that:
   - Identified the paper and its authors
   - Provided publication details
   - Used the gathered information to answer the user's question about the authors' current workplaces

6. **Termination**: The agent determined it had sufficient information to answer the query and did not make additional tool calls, ending the execution cycle.

**Important Note**: While the agent technically made both tool calls simultaneously, this wasn't the most logical approach for this specific query. A more efficient strategy would have been:
1. First search Arxiv to find the paper and identify the authors
2. Then use Tavily to search for each author's current workplace

The agent's approach of searching both tools simultaneously suggests it was trying to gather general information about the paper rather than following the logical sequence implied by "then search each of the authors." This highlights a limitation in the agent's planning capabilities - it can make multiple tool calls at once, but doesn't always sequence them optimally for complex multi-step tasks.


# 🤝 Breakout Room #2


## Part 1: LangSmith Evaluator


### Pre-processing for LangSmith


To do a little bit more preprocessing, let's wrap our LangGraph agent in a simple chain.


In [18]:
def convert_inputs(input_object):
  return {"messages" : [HumanMessage(content=input_object["text"])]}

def parse_output(input_state):
  return {"answer" : input_state["messages"][-1].content}

agent_chain_with_formatting = convert_inputs | simple_agent_graph | parse_output

agent_chain_with_formatting.invoke({"text" : "What is Deep Research?"})

{'answer': 'Deep Research typically refers to an in-depth and comprehensive investigation or analysis into a specific topic, subject, or field. It involves gathering detailed information, examining various sources, and analyzing data thoroughly to gain a profound understanding. Deep Research is often used in academic, scientific, technological, and business contexts to develop insights, inform decision-making, or advance knowledge.\n\nWould you like a more specific definition related to a particular industry or context?'}

### Task 1: Creating An Evaluation Dataset

Just as we saw last week, we'll want to create a dataset to test our Agent's ability to answer questions.

In order to do this - we'll want to provide some questions and some answers. Let's look at how we can create such a dataset below.

```python
questions = [
    {
        "inputs" : {"text" : "Who were the main authors on the 'A Comprehensive Survey of Deep Research: Systems, Methodologies, and Applications' paper?"},
        "outputs" : {"must_mention" : ["Peng", "Xu"]}
    },
    ...,
    {
        "inputs" : {"text" : "Where do the authors of the 'A Comprehensive Survey of Deep Research: Systems, Methodologies, and Applications' work now?"},
        "outputs" : {"must_mention" : ["Zhejiang", "Liberty Mutual"]}
    }
]
```


#### 🏗️ Activity #3:

Please create a dataset in the above format with at least 5 questions that pertain to the cohort use-case (more information [here](https://www.notion.so/Session-4-RAG-with-LangGraph-OSS-Local-Models-Eval-w-LangSmith-26acd547af3d80838d5beba464d7e701#26acd547af3d81d08809c9c82a462bdd)), or the use-case you're hoping to tackle in your Demo Day project.


In [25]:
questions = [   
    {
        "inputs" : {"text" : "What are the most effective AI use cases for small businesses with limited technical resources?"},
        "outputs" : {"must_mention" : ["automation", "customer service", "ROI", "small business"]}
    },
    {
        "inputs" : {"text" : "How can a marketing agency use AI to improve their client campaigns and increase efficiency?"},
        "outputs" : {"must_mention" : ["content generation", "analytics", "personalization", "marketing"]}
    },
    {
        "inputs" : {"text" : "What AI projects should a healthcare startup prioritize to maximize patient value and regulatory compliance?"},
        "outputs" : {"must_mention" : ["patient care", "compliance", "healthcare", "startup"]}
    },
    {
        "inputs" : {"text" : "What are the top AI automation opportunities for e-commerce companies in 2025?"},
        "outputs" : {"must_mention" : ["inventory", "recommendations", "chatbots", "e-commerce"]}
    },
    {
        "inputs" : {"text" : "How can non-profit organizations leverage AI to better serve their communities with minimal budget?"},
        "outputs" : {"must_mention" : ["community", "non-profit", "budget", "volunteer"]}
    },
    {
        "inputs" : {"text" : "What AI tools should a freelance consultant implement to scale their practice and serve more clients?"},
        "outputs" : {"must_mention" : ["productivity", "client management", "freelance", "scaling"]}
    },
    {
        "inputs" : {"text" : "What are the most promising AI applications for local government to improve citizen services?"},
        "outputs" : {"must_mention" : ["citizen services", "government", "public sector", "efficiency"]}
    },
    {
        "inputs" : {"text" : "How can a manufacturing company use AI to reduce costs and improve quality control?"},
        "outputs" : {"must_mention" : ["predictive maintenance", "quality control", "manufacturing", "cost reduction"]}
    },
    {
        "inputs" : {"text" : "What AI projects would provide the highest ROI for a financial services firm?"},
        "outputs" : {"must_mention" : ["fraud detection", "risk assessment", "financial services", "ROI"]}
    },
    {
        "inputs" : {"text" : "What are the best AI use cases for educational institutions to enhance student learning outcomes?"},
        "outputs" : {"must_mention" : ["personalized learning", "assessment", "education", "student outcomes"]}
    }
]

Now we can add our dataset to our LangSmith project using the following code which we saw last Thursday!


In [26]:
from langsmith import Client

client = Client()

dataset_name = f"Simple Search Agent - Evaluation Dataset - {uuid4().hex[0:8]}"

dataset = client.create_dataset(
    dataset_name=dataset_name,
    description="Questions about the cohort use-case to evaluate the Simple Search Agent."
)

client.create_examples(
    dataset_id=dataset.id,
    examples=questions
)

{'example_ids': ['c6fdc695-207e-4808-a04d-0ac4e04701c3',
  'bafd0613-5ebb-4ce4-a302-bcdcd1250df7',
  '995459cc-aef6-4bdf-96b7-e92d153ddbca',
  'f37dd62c-4310-4a0c-afd9-954cda03e87d',
  '8e2263e8-c070-4154-9ff5-94999537ac52',
  'f23c15e8-ef75-4aa0-8b42-a370fa099834',
  'cc6595d8-5994-410a-b5f4-ed799f032bc5',
  '9b93fcd4-036f-40f2-9d5b-7a3f5789f5ff',
  '5d11ea47-9752-43fb-8302-d35f424c655a',
  '20ed1cce-937c-4927-958a-60786d17902c'],
 'count': 10}

### Task 2: Adding Evaluators

Let's use the OpenEvals library to product an evaluator that we can then pass into LangSmith!

> NOTE: Examine the `CORRECTNESS_PROMPT` below!


In [27]:
from openevals.prompts import CORRECTNESS_PROMPT
print(CORRECTNESS_PROMPT)

You are an expert data labeler evaluating model outputs for correctness. Your task is to assign a score based on the following rubric:

<Rubric>
  A correct answer:
  - Provides accurate and complete information
  - Contains no factual errors
  - Addresses all parts of the question
  - Is logically consistent
  - Uses precise and accurate terminology

  When scoring, you should penalize:
  - Factual errors or inaccuracies
  - Incomplete or partial answers
  - Misleading or ambiguous statements
  - Incorrect terminology
  - Logical inconsistencies
  - Missing key information
</Rubric>

<Instructions>
  - Carefully read the input and output
  - Check for factual accuracy and completeness
  - Focus on correctness of information rather than style or verbosity
</Instructions>

<Reminder>
  The goal is to evaluate factual correctness and completeness of the response.
</Reminder>

<input>
{inputs}
</input>

<output>
{outputs}
</output>

Use the reference outputs below to help you evaluate the

In [28]:
from openevals.llm import create_llm_as_judge

correctness_evaluator = create_llm_as_judge(
        prompt=CORRECTNESS_PROMPT,
        model="openai:o3-mini", # very impactful to the final score
        feedback_key="correctness",
    )

Let's also create a custom Evaluator for our created dataset above - we do this by first making a simple Python function!


In [29]:
def must_mention(inputs: dict, outputs: dict, reference_outputs: dict) -> float:
  # determine if the phrases in the reference_outputs are in the outputs
  required = reference_outputs.get("must_mention") or []
  score = all(phrase in outputs["answer"] for phrase in required)
  return score

#### ❓ Question #4:

What are some ways you could improve this metric as-is?

> NOTE: Alternatively you can suggest where gaps exist in this method.

##### ✅ Answer:

The current `must_mention` metric has several limitations and can be improved in multiple ways:

**Current Limitations:**
1. **Binary scoring only** - It's either 1 (all phrases found) or 0 (any phrase missing), which doesn't capture partial success
2. **Case sensitivity** - The current implementation is case-sensitive, missing variations like "ROI" vs "roi"
3. **Exact string matching** - Doesn't handle synonyms, paraphrasing, or related concepts
4. **No context awareness** - Doesn't consider if the phrases are used appropriately in context
5. **No semantic understanding** - Misses conceptually correct answers that use different terminology

**Improvements:**

1. **Partial Credit Scoring:**
   ```python
   def improved_must_mention(inputs: dict, outputs: dict, reference_outputs: dict) -> float:
       required = reference_outputs.get("must_mention") or []
       found_count = sum(1 for phrase in required if phrase.lower() in outputs["answer"].lower())
       return found_count / len(required) if required else 1.0
   ```

2. **Case-Insensitive Matching:**
   ```python
   def case_insensitive_must_mention(inputs: dict, outputs: dict, reference_outputs: dict) -> float:
       required = reference_outputs.get("must_mention") or []
       answer_lower = outputs["answer"].lower()
       score = all(phrase.lower() in answer_lower for phrase in required)
       return score
   ```

3. **Synonym and Semantic Matching:**
   ```python
   from sentence_transformers import SentenceTransformer
   import numpy as np
   
   def semantic_must_mention(inputs: dict, outputs: dict, reference_outputs: dict) -> float:
       model = SentenceTransformer('all-MiniLM-L6-v2')
       required = reference_outputs.get("must_mention") or []
       answer = outputs["answer"]
       
       # Get embeddings
       answer_embedding = model.encode([answer])
       required_embeddings = model.encode(required)
       
       # Calculate cosine similarities
       similarities = np.dot(answer_embedding, required_embeddings.T).flatten()
       threshold = 0.7  # Adjust based on testing
       
       score = np.mean(similarities > threshold)
       return score
   ```

4. **Weighted Importance:**
   ```python
   def weighted_must_mention(inputs: dict, outputs: dict, reference_outputs: dict) -> float:
       required = reference_outputs.get("must_mention") or []
       weights = reference_outputs.get("weights", [1.0] * len(required))
       
       total_weight = 0
       matched_weight = 0
       
       for phrase, weight in zip(required, weights):
           total_weight += weight
           if phrase.lower() in outputs["answer"].lower():
               matched_weight += weight
       
       return matched_weight / total_weight if total_weight > 0 else 0
   ```

5. **Context-Aware Evaluation:**
   ```python
   def context_aware_must_mention(inputs: dict, outputs: dict, reference_outputs: dict) -> float:
       required = reference_outputs.get("must_mention") or []
       answer = outputs["answer"]
       
       # Check for phrases in context (e.g., within sentences)
       sentences = answer.split('.')
       context_scores = []
       
       for phrase in required:
           phrase_found = False
           for sentence in sentences:
               if phrase.lower() in sentence.lower() and len(sentence.split()) > 3:  # Context check
                   phrase_found = True
                   break
           context_scores.append(phrase_found)
       
       return sum(context_scores) / len(required) if required else 1.0
   ```

6. **Combined Multi-Metric Approach:**
   ```python
   def comprehensive_evaluation(inputs: dict, outputs: dict, reference_outputs: dict) -> dict:
       return {
           "exact_match": must_mention(inputs, outputs, reference_outputs),
           "partial_credit": improved_must_mention(inputs, outputs, reference_outputs),
           "case_insensitive": case_insensitive_must_mention(inputs, outputs, reference_outputs),
           "semantic_similarity": semantic_must_mention(inputs, outputs, reference_outputs)
       }
   ```

**Additional Gaps:**
- **No quality assessment** - Doesn't evaluate if the information is accurate or helpful
- **No completeness check** - Doesn't verify if the answer fully addresses the question
- **No relevance scoring** - Doesn't measure how relevant the mentioned concepts are to the question
- **No factual accuracy** - Doesn't verify if the mentioned concepts are used correctly
- **No actionability** - Doesn't check if the answer provides actionable insights

The most practical improvement would be implementing partial credit scoring with case-insensitive matching, as it provides more nuanced evaluation while remaining simple to implement and understand.


Task 3: Evaluating

All that is left to do is evaluate our agent's response!


In [30]:
results = client.evaluate(
    agent_chain_with_formatting,
    data=dataset.name,
    evaluators=[correctness_evaluator, must_mention],
    experiment_prefix="simple_agent, baseline",  # optional, experiment name prefix
    description="Testing the baseline system.",  # optional, experiment description
    max_concurrency=4, # optional, add concurrency
)

View the evaluation results for experiment: 'simple_agent, baseline-2f9ccd12' at:
https://smith.langchain.com/o/59e7f291-de60-42a7-a056-0fe9209518e1/datasets/93b87820-5dbc-4b33-89df-e50e048dbe87/compare?selectedSessions=1b4cd6c4-a319-4c4f-b925-8c18a3415a90




0it [00:00, ?it/s]

## Part 2: LangGraph with Helpfulness:


### Task 3: Adding Helpfulness Check and "Loop" Limits

Now that we've done evaluation - let's see if we can add an extra step where we review the content we've generated to confirm if it fully answers the user's query!

We're going to make a few key adjustments to account for this:

1. We're going to add an artificial limit on how many "loops" the agent can go through - this will help us to avoid the potential situation where we never exit the loop.
2. We'll add to our existing conditional edge to obtain the behaviour we desire.


First, let's define our state again - we can check the length of the state object, so we don't need additional state for this.


In [31]:
class AgentState(TypedDict):
  messages: Annotated[list, add_messages]

Now we can set our graph up! This process will be almost entirely the same - with the inclusion of one additional node/conditional edge!


#### 🏗️ Activity #4:

Please write markdown for the following cells to explain what each is doing.



Here, we're initializing a graph with the `AgentState` object. Then we're adding the `agent` and `action` nodes to the graph. The `agent` node can call the model, whereas the `action` node can call a tool.    

In [37]:
graph_with_helpfulness_check = StateGraph(AgentState)

graph_with_helpfulness_check.add_node("agent", call_model)
graph_with_helpfulness_check.add_node("action", tool_node)

Next, we add our entrypoint. All the entrypoint does is indicate which node is called first, which is the `agent` node in this case.

In [38]:
graph_with_helpfulness_check.set_entry_point("agent")

Next, we build a conditional edge which uses the output state of a node to determine which path to follow:
- If state contains more than 10 messages, terminate the loop 
- If the most recent message stored in state contains a tool call, route to the action node.
  - Otherwise, compute the helpfulness score using the initial query and the final response, with gpt-4.1-mini as the chat model  
    - If the response includes Y, terminate the loop as the response is determined to be helpful.
      - Otherwise, go to the "continue" node (which is just the agent node in the implementation below)


In [41]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

def tool_call_or_helpful(state):
  last_message = state["messages"][-1]

  if last_message.tool_calls:
    return "action"

  initial_query = state["messages"][0]
  final_response = state["messages"][-1]

  if len(state["messages"]) > 10:
    return "END"

  prompt_template = """\
  Given an initial query and a final response, determine if the final response is extremely helpful or not. Please indicate helpfulness with a 'Y' and unhelpfulness as an 'N'.

  Initial Query:
  {initial_query}

  Final Response:
  {final_response}"""

  helpfullness_prompt_template = PromptTemplate.from_template(prompt_template)

  helpfulness_check_model = ChatOpenAI(model="gpt-4.1-mini")

  helpfulness_chain = helpfullness_prompt_template | helpfulness_check_model | StrOutputParser()

  helpfulness_response = helpfulness_chain.invoke({"initial_query" : initial_query.content, "final_response" : final_response.content})

  if "Y" in helpfulness_response:
    return "end"
  else:
    return "continue"

Here, we connect the agent node to the `tool_call_or_helpful` conditional edge. `tool_call_or_helpful` outputs "continue", "action", or "end" which correspondingly map to the agent node, action node, or the END node.


In [42]:
graph_with_helpfulness_check.add_conditional_edges(
    "agent",
    tool_call_or_helpful,
    {
        "continue" : "agent",
        "action" : "action",
        "end" : END
    }
)

Here, we add an edge connecting the action node to the agent node, as we want the action node which calls tools to return its output to the agent.


In [43]:
graph_with_helpfulness_check.add_edge("action", "agent")

Finally, compile the workflow.


In [44]:
agent_with_helpfulness_check = graph_with_helpfulness_check.compile()

Now that the graph has been created an compiled, let's call it with a basic input containing a question about Deep Research Agents.

In [45]:
inputs = {"messages" : [HumanMessage(content="What are Deep Research Agents?")]}

async for chunk in agent_with_helpfulness_check.astream(inputs, stream_mode="updates"):
    for node, values in chunk.items():
        print(f"Receiving update from node: '{node}'")
        print(values["messages"])
        print("\n\n")

Receiving update from node: 'agent'
[AIMessage(content='Deep Research Agents are advanced AI systems designed to assist with in-depth research tasks. They leverage deep learning techniques and large datasets to analyze complex information, generate insights, and support decision-making across various fields such as science, technology, medicine, and more. These agents can automate literature reviews, extract relevant data from vast sources, and provide comprehensive summaries, making research more efficient and thorough. Would you like me to find more detailed or specific information about Deep Research Agents?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 93, 'prompt_tokens': 158, 'total_tokens': 251, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4.1-nano-2025-

## Part 3: LangGraph for the "Patterns" of GenAI

### Task 4: Helpfulness Check of Gen AI Pattern Descriptions

Let's ask our system about the 3 main patterns in Generative AI:

1. Context Engineering
2. Fine-tuning
3. Agents


In [46]:
patterns = ["Context Engineering", "Fine-tuning", "LLM-based agents"]

In [47]:
for pattern in patterns:
  what_is_string = f"What is {pattern} and when did it break onto the scene??"
  inputs = {"messages" : [HumanMessage(content=what_is_string)]}
  messages = agent_with_helpfulness_check.invoke(inputs)
  print(messages["messages"][-1].content)
  print("\n\n")

Context Engineering is a relatively new interdisciplinary field that focuses on designing, managing, and utilizing contextual information to improve the functionality and user experience of systems, devices, and applications. It involves understanding and manipulating the context in which interactions occur—such as location, time, user activity, and environmental factors—to create more adaptive and intelligent systems.

The concept of Context Engineering gained prominence with the rise of ubiquitous computing, the Internet of Things (IoT), and context-aware computing in the early 2000s. It became particularly notable as researchers and industry professionals recognized the importance of context in developing smarter, more responsive technologies.

While the exact moment it "broke onto the scene" is difficult to pinpoint, significant milestones include the publication of foundational research papers in the early 2000s and the increasing adoption of context-aware systems in commercial ap